In [1]:
# ============================================================
# GPT-2 LEGAL DOMAIN FINE-TUNING
# ============================================================
#
# MODEL:
#   openai-community/gpt2
#   GPT-2 Small / 124M parameters
#
# DATA:
#   Same InLegal dataset used in the previous notebook
#
# SPLIT:
#   90% TRAIN
#   10% VALIDATION
#
# OBJECTIVE:
#   Causal Language Modeling / Continued Pretraining
#
# IMPORTANT:
#   Validation data is NEVER used for training.
#
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q -U transformers datasets accelerate huggingface_hub


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gc
import math
import random
import shutil
import zipfile
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

from huggingface_hub import hf_hub_download


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 61

set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. ENVIRONMENT
# ============================================================

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(
        "GPU memory      :",
        round(gpu_memory, 2),
        "GB"
    )


# ============================================================
# 5. MODEL CONFIGURATION
# ============================================================

MODEL_NAME = "openai-community/gpt2"

BLOCK_SIZE = 1024


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

# Full 90% training data

MAX_TRAIN_BLOCKS = 100_000

# Use a substantial validation subset.
#
# Your previous notebook used 2,000 validation blocks.
# We use 2,000 again initially because evaluating the
# complete ~36k validation blocks is very expensive on T4.

MAX_VALID_BLOCKS = 2_000


# ------------------------------------------------------------
# Fine-tuning parameters
# ------------------------------------------------------------

NUM_EPOCHS = 2

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

PER_DEVICE_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "gpt2_legal_finetune"
)

FINAL_MODEL_DIR = (
    "/kaggle/working/"
    "gpt2_legal_finetune_final"
)

RESULTS_PATH = (
    "/kaggle/working/"
    "gpt2_legal_finetune_results.csv"
)


# ============================================================
# 6. DATASET DOWNLOAD
# ============================================================

print("\n" + "=" * 70)
print("DOWNLOADING INLEGAL DATASET")
print("=" * 70)


HF_REPO = "L-NLProc/InLegalLlama-training-data"

HF_FILENAME = "For CPT/Train_data.zip"

DATA_DIR = "/kaggle/working/legal_data"

ZIP_PATH = "/kaggle/working/Train_data.zip"


os.makedirs(
    DATA_DIR,
    exist_ok=True
)


try:

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset"
    )

    print(
        "\nDownloaded dataset publicly."
    )


except Exception as e:

    print(
        "\nPublic download failed."
    )

    print(
        "Trying Kaggle Hugging Face token..."
    )

    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()

    HF_TOKEN = user_secrets.get_secret(
        "HF-MTP-SLM"
    )

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset",
        token=HF_TOKEN
    )

    print(
        "Downloaded using HF token."
    )


shutil.copy(
    downloaded_path,
    ZIP_PATH
)


# ============================================================
# 7. EXTRACT
# ============================================================

print("\n" + "=" * 70)
print("EXTRACTING DATASET")
print("=" * 70)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zip_ref:

    zip_ref.extractall(
        DATA_DIR
    )


print(
    "Extracted to:",
    DATA_DIR
)


# ============================================================
# 8. FIND CSV
# ============================================================

csv_files = list(
    Path(DATA_DIR).rglob("*.csv")
)


print("\nCSV files found:")

for f in csv_files:

    print(
        " -",
        f
    )


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No CSV found."
    )


CSV_PATH = (
    "/kaggle/working/legal_data/"
    "Train_data/"
    "train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv"
)


if not os.path.exists(
    CSV_PATH
):

    print(
        "\nExpected CSV path not found."
    )

    print(
        "Using first CSV found."
    )

    CSV_PATH = str(
        csv_files[0]
    )


print(
    "\nUsing CSV:"
)

print(
    CSV_PATH
)


# ============================================================
# 9. LOAD DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING RAW LEGAL DATA")
print("=" * 70)


df = pd.read_csv(
    CSV_PATH
)


print(
    "Original shape:",
    df.shape
)


print(
    "Columns:",
    df.columns.tolist()
)


if "text" not in df.columns:

    raise ValueError(
        "'text' column not found."
    )


# Keep only text

df = df[
    ["text"]
].copy()


# Convert to string

df["text"] = (
    df["text"]
    .astype(str)
)


# Remove empty documents

df = df[
    df["text"].str.strip() != ""
]


# ============================================================
# 10. REMOVE EXACT DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("REMOVING EXACT DUPLICATES")
print("=" * 70)


before_duplicates = len(
    df
)


df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(
    drop=True
)


after_duplicates = len(
    df
)


print(
    "Before:",
    before_duplicates
)

print(
    "Duplicates removed:",
    before_duplicates
    - after_duplicates
)

print(
    "After:",
    after_duplicates
)


# ============================================================
# 11. 90 / 10 DOCUMENT-LEVEL SPLIT
# ============================================================

print("\n" + "=" * 70)
print("90 / 10 TRAIN / VALIDATION SPLIT")
print("=" * 70)


dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)


split_dataset = (
    dataset.train_test_split(
        test_size=0.10,
        seed=SEED
    )
)


train_dataset = (
    split_dataset["train"]
)

valid_dataset = (
    split_dataset["test"]
)


print(
    "Training documents:",
    len(train_dataset)
)

print(
    "Validation documents:",
    len(valid_dataset)
)


# ============================================================
# 12. LOAD GPT-2 TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING GPT-2 TOKENIZER")
print("=" * 70)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_NAME
    )
)


# GPT-2 does not have a separate PAD token

if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


print(
    "Tokenizer:",
    MODEL_NAME
)

print(
    "Vocabulary:",
    len(tokenizer)
)

print(
    "EOS token:",
    tokenizer.eos_token
)

print(
    "EOS ID:",
    tokenizer.eos_token_id
)

print(
    "PAD token:",
    tokenizer.pad_token
)

print(
    "PAD ID:",
    tokenizer.pad_token_id
)


# ============================================================
# 13. TOKENIZE TRAINING DATA
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZING TRAINING DATA")
print("=" * 70)


def tokenize_function(examples):

    # Add EOS at the end of every document.
    #
    # This prevents the model from treating the end of one
    # legal document as naturally continuing into another.

    texts = [
        text + tokenizer.eos_token
        for text in examples["text"]
    ]

    return tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False
    )


tokenized_train = train_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing training documents"
)


print(
    "Training tokenization complete."
)


# ============================================================
# 14. TOKENIZE VALIDATION DATA
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZING VALIDATION DATA")
print("=" * 70)


tokenized_valid = valid_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing validation documents"
)


print(
    "Validation tokenization complete."
)


# ============================================================
# 15. CREATE 1024 TOKEN BLOCKS
# ============================================================

print("\n" + "=" * 70)
print("CREATING 1024-TOKEN BLOCKS")
print("=" * 70)


def group_texts(examples):

    concatenated = {}

    for key in examples.keys():

        concatenated[key] = sum(
            examples[key],
            []
        )


    total_length = len(
        concatenated["input_ids"]
    )


    # Keep complete blocks only

    total_length = (
        total_length
        // BLOCK_SIZE
    ) * BLOCK_SIZE


    result = {

        key: [

            tokens[
                i:i + BLOCK_SIZE
            ]

            for i in range(
                0,
                total_length,
                BLOCK_SIZE
            )
        ]

        for key, tokens
        in concatenated.items()
    }


    return result


# ------------------------------------------------------------
# TRAIN BLOCKS
# ------------------------------------------------------------

lm_train_full = (
    tokenized_train.map(

        group_texts,

        batched=True,

        batch_size=1000,

        num_proc=2,

        desc="Grouping training tokens"
    )
)


print(
    "\nFull training blocks:",
    len(lm_train_full)
)


# ------------------------------------------------------------
# VALIDATION BLOCKS
# ------------------------------------------------------------

lm_valid_full = (
    tokenized_valid.map(

        group_texts,

        batched=True,

        batch_size=1000,

        num_proc=2,

        desc="Grouping validation tokens"
    )
)


print(
    "Full validation blocks:",
    len(lm_valid_full)
)


# ============================================================
# 16. SELECT TRAINING / VALIDATION BLOCKS
# ============================================================

print("\n" + "=" * 70)
print("SELECTING EXPERIMENT DATA")
print("=" * 70)


# ------------------------------------------------------------
# Shuffle training blocks before selecting.
#
# This is important because otherwise selecting the first
# 100,000 blocks could bias the experiment toward the
# beginning of the dataset.
# ------------------------------------------------------------

lm_train_full = (
    lm_train_full.shuffle(
        seed=SEED
    )
)


train_blocks_to_use = min(

    MAX_TRAIN_BLOCKS,

    len(lm_train_full)
)


lm_train = (
    lm_train_full.select(
        range(
            train_blocks_to_use
        )
    )
)


# ------------------------------------------------------------
# Validation
#
# DO NOT shuffle validation before selecting.
# We want a deterministic validation subset.
# ------------------------------------------------------------

valid_blocks_to_use = min(

    MAX_VALID_BLOCKS,

    len(lm_valid_full)
)


lm_valid = (
    lm_valid_full.select(
        range(
            valid_blocks_to_use
        )
    )
)


print(
    "Training blocks used:",
    len(lm_train)
)


print(
    "Validation blocks used:",
    len(lm_valid)
)


print(
    "\nTraining tokens:",
    f"{len(lm_train) * BLOCK_SIZE:,}"
)


print(
    "Validation tokens:",
    f"{len(lm_valid) * BLOCK_SIZE:,}"
)


# ============================================================
# 17. DATA SANITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("DATA SANITY CHECK")
print("=" * 70)


sample_ids = (
    lm_train[0]["input_ids"]
)


print(
    "Sample block length:",
    len(sample_ids)
)


print(
    "\nDecoded sample:"
)


print(
    tokenizer.decode(
        sample_ids[:500]
    )
)


# ============================================================
# 18. DATA COLLATOR
# ============================================================

print("\n" + "=" * 70)
print("CREATING CAUSAL LM COLLATOR")
print("=" * 70)


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False
    )
)


print(
    "Objective:",
    "Causal Language Modeling"
)

print(
    "MLM:",
    False
)


# ============================================================
# 19. LOAD PRETRAINED GPT-2
# ============================================================

print("\n" + "=" * 70)
print("LOADING PRETRAINED GPT-2")
print("=" * 70)


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)


# GPT-2 padding

model.config.pad_token_id = (
    tokenizer.pad_token_id
)


# Required when using gradient checkpointing

model.config.use_cache = False


num_parameters = sum(

    p.numel()

    for p in model.parameters()
)


trainable_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    "Model loaded successfully."
)


print(
    "\nTotal parameters:",
    f"{num_parameters:,}"
)


print(
    "Total parameters (M):",
    f"{num_parameters / 1e6:.2f}M"
)


print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# 20. TRAINING ARGUMENTS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)


print(
    "Epochs:",
    NUM_EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Batch size:",
    PER_DEVICE_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch size:",
    PER_DEVICE_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)


training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_steps=0,

    lr_scheduler_type="cosine",

    per_device_train_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    per_device_eval_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    # --------------------------------------------------------
    # Evaluate after every epoch
    # --------------------------------------------------------

    eval_strategy="epoch",

    # --------------------------------------------------------
    # Save after every epoch
    # --------------------------------------------------------

    save_strategy="epoch",

    save_total_limit=2,

    # --------------------------------------------------------
    # Automatically keep model with lowest validation loss
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=100,

    # --------------------------------------------------------
    # T4 mixed precision
    # --------------------------------------------------------

    fp16=torch.cuda.is_available(),

    # --------------------------------------------------------
    # Memory optimization
    # --------------------------------------------------------

    gradient_checkpointing=True,

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=SEED,

    data_seed=SEED,

    report_to="none",

    remove_unused_columns=True
)


# ============================================================
# 21. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=lm_train,

    eval_dataset=lm_valid,

    data_collator=data_collator,

    processing_class=tokenizer
)


print(
    "\nTrainer created successfully."
)


# ============================================================
# 22. BASELINE VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE — BEFORE FINE-TUNING")
print("=" * 70)


baseline_start = time.time()


baseline_results = (
    trainer.evaluate()
)


baseline_time = (
    time.time()
    - baseline_start
)


baseline_loss = (
    baseline_results["eval_loss"]
)


try:

    baseline_perplexity = math.exp(
        baseline_loss
    )

except OverflowError:

    baseline_perplexity = float("inf")


print(
    "\nBaseline validation loss:",
    f"{baseline_loss:.6f}"
)


print(
    "Baseline validation perplexity:",
    f"{baseline_perplexity:.6f}"
)


print(
    "Baseline evaluation time:",
    f"{baseline_time / 60:.2f} minutes"
)


# ============================================================
# 23. CLEAR CACHE
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 24. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("STARTING GPT-2 LEGAL FINE-TUNING")
print("=" * 70)


print(
    "\nTraining documents:",
    len(train_dataset)
)


print(
    "Validation documents:",
    len(valid_dataset)
)


print(
    "Training blocks:",
    len(lm_train)
)


print(
    "Validation blocks:",
    len(lm_valid)
)


print(
    "Training tokens:",
    f"{len(lm_train) * BLOCK_SIZE:,}"
)


print(
    "Validation tokens:",
    f"{len(lm_valid) * BLOCK_SIZE:,}"
)


print(
    "\nStarting training...\n"
)


training_start = time.time()


CHECKPOINT_PATH = "/kaggle/input/datasets/infinityy08/gpt-2nfinetuning-2-output-checkpoint-6250/gpt2_legal_finetune/checkpoint-6250"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(...)

train_result = trainer.train(
    resume_from_checkpoint=CHECKPOINT_PATH
)


training_time = (
    time.time()
    - training_start
)


# ============================================================
# 25. TRAINING RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)


print(
    "\nTraining time:",
    f"{training_time / 3600:.2f} hours"
)


print(
    f"Training time:",
    f"{training_time / 60:.2f} minutes"
)


print(
    "\nTraining metrics:"
)


for key, value in (
    train_result.metrics.items()
):

    if isinstance(
        value,
        float
    ):

        print(
            f"{key}: {value:.6f}"
        )

    else:

        print(
            f"{key}: {value}"
        )


# ============================================================
# 26. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION — UNSEEN 10%")
print("=" * 70)


final_eval_start = time.time()


final_results = (
    trainer.evaluate()
)


final_eval_time = (
    time.time()
    - final_eval_start
)


final_loss = (
    final_results["eval_loss"]
)


try:

    final_perplexity = math.exp(
        final_loss
    )

except OverflowError:

    final_perplexity = float("inf")


print(
    "\nFinal validation loss:",
    f"{final_loss:.6f}"
)


print(
    "Final validation perplexity:",
    f"{final_perplexity:.6f}"
)


print(
    "Final evaluation time:",
    f"{final_eval_time / 60:.2f} minutes"
)


# ============================================================
# 27. IMPROVEMENT
# ============================================================

loss_improvement = (
    baseline_loss
    - final_loss
)


if baseline_loss != 0:

    loss_improvement_percent = (

        loss_improvement
        / baseline_loss
        * 100

    )

else:

    loss_improvement_percent = 0


print("\n" + "=" * 70)
print("DOMAIN ADAPTATION IMPROVEMENT")
print("=" * 70)


print(
    "\nBaseline validation loss:",
    f"{baseline_loss:.6f}"
)


print(
    "Final validation loss:",
    f"{final_loss:.6f}"
)


print(
    "Loss improvement:",
    f"{loss_improvement:.6f}"
)


print(
    "Loss improvement (%):",
    f"{loss_improvement_percent:.2f}%"
)


print(
    "\nBaseline perplexity:",
    f"{baseline_perplexity:.6f}"
)


print(
    "Final perplexity:",
    f"{final_perplexity:.6f}"
)


# ============================================================
# 28. SAVE BEST MODEL
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINE-TUNED MODEL")
print("=" * 70)


trainer.save_model(
    FINAL_MODEL_DIR
)


tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


print(
    "\nModel saved to:"
)

print(
    FINAL_MODEL_DIR
)


# ============================================================
# 29. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame({

    "model": [
        "GPT-2"
    ],

    "model_name": [
        MODEL_NAME
    ],

    "architecture": [
        "Transformer"
    ],

    "initialization": [
        "Pretrained"
    ],

    "objective": [
        "Causal Language Modeling"
    ],

    "train_documents": [
        len(train_dataset)
    ],

    "validation_documents": [
        len(valid_dataset)
    ],

    "block_size": [
        BLOCK_SIZE
    ],

    "full_train_blocks": [
        len(lm_train_full)
    ],

    "full_validation_blocks": [
        len(lm_valid_full)
    ],

    "train_blocks_used": [
        len(lm_train)
    ],

    "validation_blocks_used": [
        len(lm_valid)
    ],

    "train_tokens_used": [
        len(lm_train)
        * BLOCK_SIZE
    ],

    "validation_tokens_used": [
        len(lm_valid)
        * BLOCK_SIZE
    ],

    "epochs": [
        NUM_EPOCHS
    ],

    "learning_rate": [
        LEARNING_RATE
    ],

    "weight_decay": [
        WEIGHT_DECAY
    ],

    "batch_size": [
        PER_DEVICE_BATCH_SIZE
    ],

    "gradient_accumulation": [
        GRADIENT_ACCUMULATION_STEPS
    ],

    "effective_batch_size": [
        PER_DEVICE_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    ],

    "baseline_validation_loss": [
        baseline_loss
    ],

    "final_validation_loss": [
        final_loss
    ],

    "baseline_perplexity": [
        baseline_perplexity
    ],

    "final_perplexity": [
        final_perplexity
    ],

    "loss_improvement": [
        loss_improvement
    ],

    "loss_improvement_percent": [
        loss_improvement_percent
    ],

    "training_time_hours": [
        training_time / 3600
    ]
})


results_df.to_csv(
    RESULTS_PATH,
    index=False
)


print(
    "\nResults saved to:"
)

print(
    RESULTS_PATH
)


# ============================================================
# 30. GENERATION TEST
# ============================================================

print("\n" + "=" * 70)
print("TESTING FINE-TUNED GPT-2")
print("=" * 70)


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = trainer.model

model.to(device)

model.eval()


prompt = (
    "The Supreme Court of India held that"
)


inputs = tokenizer(
    prompt,
    return_tensors="pt"
)


inputs = {

    key: value.to(device)

    for key, value
    in inputs.items()
}


with torch.no_grad():

    generated_ids = model.generate(

        **inputs,

        max_new_tokens=150,

        do_sample=True,

        temperature=0.7,

        top_p=0.90,

        repetition_penalty=1.1,

        pad_token_id=(
            tokenizer.eos_token_id
        )
    )


generated_text = tokenizer.decode(

    generated_ids[0],

    skip_special_tokens=True
)


print(
    "\nPROMPT:"
)

print(
    prompt
)


print(
    "\nGENERATED TEXT:"
)

print(
    generated_text
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL GPT-2 EXPERIMENT SUMMARY")
print("=" * 70)


print(
    "\nModel:",
    MODEL_NAME
)


print(
    "Train documents:",
    len(train_dataset)
)


print(
    "Validation documents:",
    len(valid_dataset)
)


print(
    "Train blocks:",
    len(lm_train)
)


print(
    "Validation blocks:",
    len(lm_valid)
)


print(
    "Train tokens:",
    f"{len(lm_train) * BLOCK_SIZE:,}"
)


print(
    "Validation tokens:",
    f"{len(lm_valid) * BLOCK_SIZE:,}"
)


print(
    "\nBaseline validation loss:",
    f"{baseline_loss:.6f}"
)


print(
    "Final validation loss:",
    f"{final_loss:.6f}"
)


print(
    "\nBaseline perplexity:",
    f"{baseline_perplexity:.6f}"
)


print(
    "Final perplexity:",
    f"{final_perplexity:.6f}"
)


print(
    "\nLoss improvement:",
    f"{loss_improvement:.6f}"
)


print(
    "Loss improvement (%):",
    f"{loss_improvement_percent:.2f}%"
)


print(
    "\nTraining time:",
    f"{training_time / 3600:.2f} hours"
)


print(
    "\nFine-tuned model:"
)

print(
    FINAL_MODEL_DIR
)


print(
    "\nResults CSV:"
)

print(
    RESULTS_PATH
)


print("\n" + "=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 79.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ENVIRONMENT
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU m

For CPT/Train_data.zip: reconstructing file:   0%|          |  0.00B /  518MB            

For CPT/Train_data.zip: downloading bytes:           |  0.00B            

Downloaded using HF token.

EXTRACTING DATASET
Extracted to: /kaggle/working/legal_data

CSV files found:
 - /kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv

Using CSV:
/kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv

LOADING RAW LEGAL DATA
Original shape: (138321, 1)
Columns: ['text']

REMOVING EXACT DUPLICATES
Before: 138321
Duplicates removed: 4665
After: 133656

90 / 10 TRAIN / VALIDATION SPLIT


Training documents: 120290
Validation documents: 13366

LOADING GPT-2 TOKENIZER


...


TRAINING COMPLETED

Training time: 8.51 hours
Training time: 510.60 minutes

Training metrics:
train_runtime: 30625.268000
train_samples_per_second: 6.531000
train_steps_per_second: 0.408000
total_flos: 104516812800000000.000000
train_loss: 1.369313
epoch: 2.000000

FINAL VALIDATION — UNSEEN 10%


Training Loss,Validation Loss,Epoch
2.730388,2.690176,2



Final validation loss: 2.690176
Final validation perplexity: 14.734276
Final evaluation time: 3.68 minutes

DOMAIN ADAPTATION IMPROVEMENT

Baseline validation loss: 3.455664
Final validation loss: 2.690176
Loss improvement: 0.765487
Loss improvement (%): 22.15%

Baseline perplexity: 31.679307
Final perplexity: 14.734276

SAVING FINE-TUNED MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/kaggle/working/gpt2_legal_finetune_final

Results saved to:
/kaggle/working/gpt2_legal_finetune_results.csv

TESTING FINE-TUNED GPT-2

PROMPT:
The Supreme Court of India held that

GENERATED TEXT:
The Supreme Court of India held that the plaintiff was entitled to a declaration and under Section 12(1)(a) he had no right.He could not get relief in terms of section 11, even if it were prayed for by him before this court or otherwise through an order passed on his behalf as per Rule 18A (2).This is clearly contrary to law laid down therein where mere application does not confer any rights upon either party but merely transfers certain powers conferred at some other stage from one person to another which are vested in both parties when such transfer takes place It cannot be said that there is absolutely nothing wrong with granting permission so long as only because petitioners did apply has been denied The view taken above shows clear conflict of views regarding grant of p